In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt

csv_path = "/Users/nico/Documents/PhD research/energy poverty/global_combined_HDD_with_EC2015.csv"
shp_path = "/Users/nico/Documents/PhD research/energy poverty/GDL Shapefiles V4/GDL Shapefiles V4.shp"

# 1. read data
df = pd.read_csv(csv_path)
gdf = gpd.read_file(shp_path)

# 2. keep one record per GDL unit for 2015
df2015 = df[df["time"].astype(str).str.startswith("2015_")].copy()
df2015 = df2015.sort_values(["GDLcode", "time"]).groupby("GDLcode", as_index=False).first()

# 3. clean
df2015["income_clean"] = df2015["income"].replace(-1, np.nan)
df2015["wealth_clean"] = df2015["wealth"].replace(-1, np.nan)

df2015 = df2015[(df2015["population"] > 0) & (df2015["EC2015_sum"] > 0)].copy()
df2015["EC2015_pc"] = df2015["EC2015_sum"] / df2015["population"]
df2015["log10_EC2015_pc"] = np.log10(df2015["EC2015_pc"])

print("Remaining GDL units:", len(df2015))
print("Countries:", df2015["country"].nunique())

print("\nEconomic variable coverage:")
print("income available:", df2015["income_clean"].notna().sum())
print("wealth available:", df2015["wealth_clean"].notna().sum())

# 4. merge to geometry
map_gdf = gdf.merge(
    df2015[[
        "GDLcode", "iso_code", "country", "region", "population",
        "EC2015_sum", "EC2015_pc", "log10_EC2015_pc",
        "income_clean", "wealth_clean", "rural", "education", "age", "Hhsize", "gender"
    ]],
    on="GDLcode",
    how="inner"
)

# 5. histogram of EC2015_pc
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(df2015["EC2015_pc"], bins=80)
ax.set_xlabel("EC2015 per capita (kWh/person)")
ax.set_ylabel("Count")
ax.set_title("Distribution of EC2015 per capita")
plt.show()

# 6. histogram of log10 EC2015_pc
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(df2015["log10_EC2015_pc"], bins=80)
ax.set_xlabel("log10(EC2015 per capita)")
ax.set_ylabel("Count")
ax.set_title("Distribution of log10 EC2015 per capita")
plt.show()

# 7. global map
fig, ax = plt.subplots(figsize=(16, 8))
map_gdf.plot(
    column="log10_EC2015_pc",
    ax=ax,
    legend=True,
    linewidth=0.05
)
ax.set_title("Global distribution of log10 per capita electricity consumption, 2015")
ax.set_axis_off()
plt.show()

In [ ]:
import country_converter as coco

cc = coco.CountryConverter()

print("valid_class:")
print(sorted(cc.valid_class))

print("\ncolumns in cc.data:")
print(cc.data.columns.tolist())

In [ ]:
import pandas as pd
import numpy as np
import country_converter as coco

csv_path = "/Users/nico/Documents/PhD research/energy poverty/global_combined_HDD_with_EC2015.csv"

# =========================
# 1. read data and keep one row per GDL in 2015
# =========================
df = pd.read_csv(csv_path)

df2015 = df[df["time"].astype(str).str.startswith("2015_")].copy()
df2015 = (
    df2015.sort_values(["GDLcode", "time"])
    .groupby("GDLcode", as_index=False)
    .first()
)

# =========================
# 2. clean sample
# =========================
df2015["income_clean"] = df2015["income"].replace(-1, np.nan)
df2015["wealth_clean"] = df2015["wealth"].replace(-1, np.nan)

df2015 = df2015[(df2015["population"] > 0) & (df2015["EC2015_sum"] > 0)].copy()

df2015["EC2015_pc"] = df2015["EC2015_sum"] / df2015["population"]
df2015["log10_EC2015_pc"] = np.log10(df2015["EC2015_pc"])

# =========================
# 3. merge country metadata
# =========================
cc = coco.CountryConverter()
meta = cc.data[["ISO3", "name_short", "continent", "UNregion"]].drop_duplicates().copy()

df2015 = df2015.merge(
    meta.rename(columns={"ISO3": "iso_code"}),
    on="iso_code",
    how="left"
)

# =========================
# 4. define UN-style analysis regions
# =========================
south_asia = {
    "AFG", "BGD", "BTN", "IND", "IRN", "LKA", "MDV", "NPL", "PAK"
}

east_se_asia = {
    "BRN", "CHN", "IDN", "JPN", "KHM", "KOR", "LAO", "MMR",
    "MNG", "MYS", "PHL", "PRK", "SGP", "THA", "TLS", "VNM"
}

north_africa_west_asia = {
    "DZA", "BHR", "EGY", "IRQ", "ISR", "JOR", "KWT", "LBN", "LBY", "MAR",
    "OMN", "PSE", "QAT", "SAU", "SYR", "TUN", "TUR", "ARE", "YEM",
    "ARM", "AZE", "CYP", "GEO", "KAZ", "KGZ", "TJK", "TKM", "UZB"
}

latin_america_caribbean = {
    "ATG", "ARG", "BHS", "BRB", "BLZ", "BOL", "BRA", "CHL", "COL", "CRI",
    "CUB", "DMA", "DOM", "ECU", "SLV", "GRD", "GTM", "GUY", "HTI", "HND",
    "JAM", "MEX", "NIC", "PAN", "PRY", "PER", "KNA", "LCA", "VCT", "SUR",
    "TTO", "URY", "VEN"
}

north_america_oceania = {
    "AUS", "CAN", "FSM", "FJI", "KIR", "MHL", "NRU", "NZL", "PLW", "PNG",
    "SLB", "TON", "TUV", "USA", "VUT", "WSM"
}

def make_region(iso, continent):
    if iso in south_asia:
        return "South Asia"
    if iso in east_se_asia:
        return "Eastern and South-eastern Asia"
    if iso in north_africa_west_asia:
        return "Northern Africa and Western Asia"
    if iso in latin_america_caribbean:
        return "Latin America and the Caribbean"
    if iso in north_america_oceania:
        return "Northern America and Oceania"
    if continent == "Africa":
        return "Sub-Saharan Africa"
    if continent == "Europe":
        return "Europe"
    return np.nan

df2015["analysis_region"] = [
    make_region(i, c) for i, c in zip(df2015["iso_code"], df2015["continent"])
]

# =========================
# 5. quick checks
# =========================
print("Cleaned sample size:", len(df2015))
print("Countries:", df2015["country"].nunique())

print("\nAnalysis region counts:")
print(df2015["analysis_region"].value_counts(dropna=False))

print("\nUnmapped countries:")
print(
    df2015.loc[df2015["analysis_region"].isna(), ["iso_code", "country"]]
    .drop_duplicates()
    .sort_values("iso_code")
)

# 可选保存
out_path = "/Users/nico/Documents/PhD research/energy poverty/analysis_2015_region_ready.csv"
df2015.to_csv(out_path, index=False)
print("\nSaved to:", out_path)

In [ ]:
import pandas as pd
import numpy as np

csv_path = "/Users/nico/Documents/PhD research/energy poverty/analysis_2015_region_ready.csv"
df2015 = pd.read_csv(csv_path)

# 去掉 analysis_region 缺失
df2015 = df2015[df2015["analysis_region"].notna()].copy()

# tertile helper
def add_tertile_group(df_in, value_col, by_col, out_col):
    df_out = df_in.copy()

    def cut_one_group(s):
        out = pd.Series(pd.NA, index=s.index, dtype="object")
        mask = s.notna()

        if mask.sum() < 6 or s[mask].nunique() < 3:
            return out

        ranked = s[mask].rank(method="first")
        bins = pd.qcut(ranked, 3, labels=["low", "middle", "high"])
        out.loc[mask] = bins.astype(str).values
        return out

    df_out[out_col] = (
        df_out.groupby(by_col, group_keys=False)[value_col]
        .apply(cut_one_group)
    )
    return df_out

# 所有属性
attrs = {
    "income_clean": "income",
    "wealth_clean": "wealth",
    "rural": "rural",
    "education": "education",
    "age": "age",
    "Hhsize": "hhsize",
    "gender": "gender"
}

# 逐个属性做区域内 tertile
for col, short in attrs.items():
    df2015 = add_tertile_group(
        df2015,
        value_col=col,
        by_col="analysis_region",
        out_col=f"{short}_tertile_region"
    )

# 检查结果
print("Region counts:")
print(df2015["analysis_region"].value_counts())

print("\nTertile coverage by attribute:")
for col, short in attrs.items():
    print(f"\n{short}:")
    print(df2015[f"{short}_tertile_region"].value_counts(dropna=False))

# 保存
out_path = "/Users/nico/Documents/PhD research/energy poverty/analysis_2015_region_tertiles.csv"
df2015.to_csv(out_path, index=False)
print("\nSaved to:", out_path)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =========================
# 0. paths
# =========================
in_path = "/Users/nico/Documents/PhD research/energy poverty/analysis_2015_region_tertiles.csv"
out_dir = "/Users/nico/Documents/PhD research/energy poverty/region_theil_outputs"
os.makedirs(out_dir, exist_ok=True)

# =========================
# 1. read data
# =========================
df = pd.read_csv(in_path)

# 只保留区域非空、权重和 outcome 有效
df = df[df["analysis_region"].notna()].copy()
df = df[(df["population"] > 0) & (df["EC2015_pc"] > 0)].copy()

# 属性与对应 tertile 列
attr_map = {
    "income": "income_tertile_region",
    "wealth": "wealth_tertile_region",
    "rural": "rural_tertile_region",
    "education": "education_tertile_region",
    "age": "age_tertile_region",
    "hhsize": "hhsize_tertile_region",
    "gender": "gender_tertile_region"
}

region_order = [
    "Sub-Saharan Africa",
    "Northern Africa and Western Asia",
    "South Asia",
    "Eastern and South-eastern Asia",
    "Europe",
    "Latin America and the Caribbean",
    "Northern America and Oceania"
]

group_order = ["low", "middle", "high"]

# =========================
# 2. functions
# =========================
def weighted_theil_t(y, w):
    y = np.asarray(y, dtype=float)
    w = np.asarray(w, dtype=float)

    mask = np.isfinite(y) & np.isfinite(w) & (y > 0) & (w > 0)
    y = y[mask]
    w = w[mask]

    if len(y) == 0:
        return np.nan

    W = w.sum()
    mu = np.sum(w * y) / W
    ratio = y / mu
    return np.sum((w / W) * ratio * np.log(ratio))

def theil_decompose_by_group(sub, outcome_col, weight_col, group_col):
    # 只保留该属性分组非空的样本
    d = sub[[outcome_col, weight_col, group_col]].copy()
    d = d[d[group_col].notna()].copy()
    d = d[(d[outcome_col] > 0) & (d[weight_col] > 0)].copy()

    if len(d) == 0:
        return None, None

    y = d[outcome_col].values
    w = d[weight_col].values
    g = d[group_col].astype(str).values

    W = w.sum()
    mu = np.sum(w * y) / W

    overall = weighted_theil_t(y, w)

    group_rows = []
    between = 0.0
    within = 0.0

    for grp, dg in d.groupby(group_col):
        yg = dg[outcome_col].values
        wg = dg[weight_col].values

        Wg = wg.sum()
        mug = np.sum(wg * yg) / Wg
        Tg = weighted_theil_t(yg, wg)

        between += (Wg / W) * (mug / mu) * np.log(mug / mu)
        within += (Wg / W) * (mug / mu) * Tg

        group_rows.append({
            "group": grp,
            "sample_units": len(dg),
            "population_sum": Wg,
            "group_mean_weighted": mug,
            "group_median_unweighted": dg[outcome_col].median(),
            "group_theil_within": Tg
        })

    group_df = pd.DataFrame(group_rows)

    result = {
        "sample_units": len(d),
        "population_sum": W,
        "overall": overall,
        "between": between,
        "within": within,
        "between_share": between / overall if pd.notna(overall) and overall > 0 else np.nan,
        "within_share": within / overall if pd.notna(overall) and overall > 0 else np.nan
    }
    return result, group_df

# =========================
# 3. compute decomposition for every region × attribute
# =========================
result_rows = []
group_summary_rows = []

for region in region_order:
    sub_region = df[df["analysis_region"] == region].copy()
    if len(sub_region) == 0:
        continue

    for attr, group_col in attr_map.items():
        result, group_df = theil_decompose_by_group(
            sub=sub_region,
            outcome_col="EC2015_pc",
            weight_col="population",
            group_col=group_col
        )

        if result is None:
            result_rows.append({
                "analysis_region": region,
                "attribute": attr,
                "sample_units": 0,
                "population_sum": np.nan,
                "overall": np.nan,
                "between": np.nan,
                "within": np.nan,
                "between_share": np.nan,
                "within_share": np.nan
            })
            continue

        result_rows.append({
            "analysis_region": region,
            "attribute": attr,
            **result
        })

        group_df["analysis_region"] = region
        group_df["attribute"] = attr
        group_summary_rows.append(group_df)

results_df = pd.DataFrame(result_rows)
group_summary_df = pd.concat(group_summary_rows, ignore_index=True)

# 排序
results_df["analysis_region"] = pd.Categorical(results_df["analysis_region"], categories=region_order, ordered=True)
results_df["attribute"] = pd.Categorical(results_df["attribute"], categories=list(attr_map.keys()), ordered=True)
results_df = results_df.sort_values(["analysis_region", "attribute"]).reset_index(drop=True)

group_summary_df["analysis_region"] = pd.Categorical(group_summary_df["analysis_region"], categories=region_order, ordered=True)
group_summary_df["attribute"] = pd.Categorical(group_summary_df["attribute"], categories=list(attr_map.keys()), ordered=True)
group_summary_df["group"] = pd.Categorical(group_summary_df["group"], categories=group_order, ordered=True)
group_summary_df = group_summary_df.sort_values(["analysis_region", "attribute", "group"]).reset_index(drop=True)

# =========================
# 4. save tables
# =========================
results_csv = os.path.join(out_dir, "region_attribute_theil_decomposition.csv")
group_csv = os.path.join(out_dir, "region_attribute_group_summary.csv")

results_df.to_csv(results_csv, index=False)
group_summary_df.to_csv(group_csv, index=False)

# =========================
# 5. Jupyter display: only BETWEEN layer
# =========================
between_pivot = results_df.pivot(index="analysis_region", columns="attribute", values="between_share")
between_pivot = between_pivot.loc[region_order, list(attr_map.keys())]

print("Between-share table:")
display(between_pivot.round(3))

# =========================
# 6. visualization 1: heatmap of between_share
# =========================
fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(between_pivot.values, aspect="auto")

ax.set_xticks(range(len(between_pivot.columns)))
ax.set_xticklabels(between_pivot.columns, rotation=45, ha="right")
ax.set_yticks(range(len(between_pivot.index)))
ax.set_yticklabels(between_pivot.index)

for i in range(between_pivot.shape[0]):
    for j in range(between_pivot.shape[1]):
        val = between_pivot.iloc[i, j]
        txt = "" if pd.isna(val) else f"{val:.2f}"
        ax.text(j, i, txt, ha="center", va="center", fontsize=9)

ax.set_title("Between-group share of electricity inequality by region and attribute")
cbar = plt.colorbar(im, ax=ax)
cbar.set_label("between_share")
plt.tight_layout()

heatmap_path = os.path.join(out_dir, "region_between_share_heatmap.png")
plt.savefig(heatmap_path, dpi=300, bbox_inches="tight")
plt.show()

# =========================
# 7. visualization 2: grouped bar chart of between_share
# =========================
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(region_order))
attrs = list(attr_map.keys())
bar_w = 0.11

for k, attr in enumerate(attrs):
    vals = between_pivot[attr].values
    ax.bar(x + (k - (len(attrs)-1)/2) * bar_w, vals, width=bar_w, label=attr)

ax.set_xticks(x)
ax.set_xticklabels(region_order, rotation=45, ha="right")
ax.set_ylabel("between_share")
ax.set_title("Between-group share by region across attributes")
ax.legend(ncol=4, frameon=False)
plt.tight_layout()

bar_path = os.path.join(out_dir, "region_between_share_groupedbar.png")
plt.savefig(bar_path, dpi=300, bbox_inches="tight")
plt.show()

# =========================
# 8. optional: save one figure per attribute
# =========================
for attr in attr_map.keys():
    fig, ax = plt.subplots(figsize=(8, 4.5))
    sub = results_df[results_df["attribute"] == attr].copy()
    sub = sub.set_index("analysis_region").reindex(region_order).reset_index()

    ax.bar(sub["analysis_region"], sub["between_share"])
    ax.set_title(f"Between-group share by region: {attr}")
    ax.set_ylabel("between_share")
    ax.tick_params(axis="x", rotation=45)
    plt.tight_layout()

    fpath = os.path.join(out_dir, f"region_between_share_{attr}.png")
    plt.savefig(fpath, dpi=300, bbox_inches="tight")
    plt.close(fig)

print("\nSaved files:")
print(results_csv)
print(group_csv)
print(heatmap_path)
print(bar_path)
print(f"Attribute-specific figures saved in: {out_dir}")

In [ ]:
import pandas as pd
import numpy as np

csv_path = "/Users/nico/Documents/PhD research/energy poverty/analysis_2015_region_tertiles.csv"
df = pd.read_csv(csv_path)

df = df[df["analysis_region"].notna()].copy()
df = df[(df["population"] > 0) & (df["EC2015_pc"] > 0)].copy()

# 按国家统计
rows = []
for country, sub in df.groupby("country"):
    rows.append({
        "analysis_region": sub["analysis_region"].iloc[0],
        "country": country,
        "gdl_units": len(sub),
        "income_n": sub["income_clean"].notna().sum(),
        "wealth_n": sub["wealth_clean"].notna().sum(),
        "rural_n": sub["rural"].notna().sum(),
        "education_n": sub["education"].notna().sum(),
        "age_n": sub["age"].notna().sum(),
        "hhsize_n": sub["Hhsize"].notna().sum(),
        "gender_n": sub["gender"].notna().sum(),
    })

country_avail = pd.DataFrame(rows)

# 通用属性都可用
country_avail["core_ok"] = (
    (country_avail["rural_n"] >= 6) &
    (country_avail["education_n"] >= 6) &
    (country_avail["age_n"] >= 6) &
    (country_avail["hhsize_n"] >= 6) &
    (country_avail["gender_n"] >= 6)
)

# 经济属性可用
country_avail["income_ok"] = country_avail["income_n"] >= 6
country_avail["wealth_ok"] = country_avail["wealth_n"] >= 6

print("Core-attribute countries:")
print(
    country_avail.loc[country_avail["core_ok"], ["analysis_region", "country", "gdl_units"]]
    .sort_values(["analysis_region", "gdl_units"], ascending=[True, False])
    .to_string(index=False)
)

print("\nIncome countries:")
print(
    country_avail.loc[country_avail["income_ok"], ["analysis_region", "country", "gdl_units", "income_n"]]
    .sort_values(["analysis_region", "gdl_units"], ascending=[True, False])
    .to_string(index=False)
)

print("\nWealth countries:")
print(
    country_avail.loc[country_avail["wealth_ok"], ["analysis_region", "country", "gdl_units", "wealth_n"]]
    .sort_values(["analysis_region", "gdl_units"], ascending=[True, False])
    .to_string(index=False)
)

out_path = "/Users/nico/Documents/PhD research/energy poverty/country_attribute_availability.csv"
country_avail.to_csv(out_path, index=False)
print("\nSaved to:", out_path)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =========================
# 0. paths
# =========================
in_path = "/Users/nico/Documents/PhD research/energy poverty/analysis_2015_region_ready.csv"
base_out_dir = "/Users/nico/Documents/PhD research/energy poverty/ses_theil_outputs_baronly"
os.makedirs(base_out_dir, exist_ok=True)

region_out_dir = os.path.join(base_out_dir, "region")
country_out_dir = os.path.join(base_out_dir, "country")
os.makedirs(region_out_dir, exist_ok=True)
os.makedirs(country_out_dir, exist_ok=True)

# =========================
# 1. read data
# =========================
df = pd.read_csv(in_path)
df = df[df["analysis_region"].notna()].copy()
df = df[(df["population"] > 0) & (df["EC2015_pc"] > 0)].copy()

# =========================
# 2. build SES proxy
# income first, otherwise wealth
# =========================
df["ses_value"] = df["income_clean"]
df.loc[df["ses_value"].isna(), "ses_value"] = df.loc[df["ses_value"].isna(), "wealth_clean"]

df["ses_source"] = pd.Series(np.nan, index=df.index, dtype="object")
df.loc[df["income_clean"].notna(), "ses_source"] = "income"
df.loc[df["income_clean"].isna() & df["wealth_clean"].notna(), "ses_source"] = "wealth"

# =========================
# 3. helper functions
# =========================
def weighted_theil_t(y, w):
    y = np.asarray(y, dtype=float)
    w = np.asarray(w, dtype=float)
    mask = np.isfinite(y) & np.isfinite(w) & (y > 0) & (w > 0)
    y = y[mask]
    w = w[mask]
    if len(y) == 0:
        return np.nan
    W = w.sum()
    mu = np.sum(w * y) / W
    ratio = y / mu
    return np.sum((w / W) * ratio * np.log(ratio))

def add_country_tertiles(df_in, value_col, out_col):
    df_out = df_in.copy()

    def cut_one_country(s):
        out = pd.Series(pd.NA, index=s.index, dtype="object")
        mask = s.notna()
        if mask.sum() < 6 or s[mask].nunique() < 3:
            return out
        ranked = s[mask].rank(method="first")
        bins = pd.qcut(ranked, 3, labels=["low", "middle", "high"])
        out.loc[mask] = bins.astype(str).values
        return out

    df_out[out_col] = (
        df_out.groupby("country", group_keys=False)[value_col]
        .apply(cut_one_country)
    )
    return df_out

def theil_decompose_by_group(sub, outcome_col, weight_col, group_col):
    d = sub[[outcome_col, weight_col, group_col]].copy()
    d = d[d[group_col].notna()].copy()
    d = d[(d[outcome_col] > 0) & (d[weight_col] > 0)].copy()

    if len(d) == 0:
        return None, None

    y = d[outcome_col].values
    w = d[weight_col].values
    W = w.sum()
    mu = np.sum(w * y) / W
    overall = weighted_theil_t(y, w)

    group_rows = []
    between = 0.0
    within = 0.0

    for grp, dg in d.groupby(group_col):
        yg = dg[outcome_col].values
        wg = dg[weight_col].values
        Wg = wg.sum()
        mug = np.sum(wg * yg) / Wg
        Tg = weighted_theil_t(yg, wg)

        between += (Wg / W) * (mug / mu) * np.log(mug / mu)
        within += (Wg / W) * (mug / mu) * Tg

        group_rows.append({
            "group": grp,
            "sample_units": len(dg),
            "population_sum": Wg,
            "group_mean_weighted": mug,
            "group_median_unweighted": dg[outcome_col].median(),
            "group_theil_within": Tg
        })

    group_df = pd.DataFrame(group_rows)
    result = {
        "sample_units": len(d),
        "population_sum": W,
        "overall": overall,
        "between": between,
        "within": within,
        "between_share": between / overall if pd.notna(overall) and overall > 0 else np.nan,
        "within_share": within / overall if pd.notna(overall) and overall > 0 else np.nan
    }
    return result, group_df

# =========================
# 4. build country tertiles
# =========================
df = add_country_tertiles(df, "ses_value", "ses_tertile_country")
df = add_country_tertiles(df, "rural", "rural_tertile_country")
df = add_country_tertiles(df, "education", "education_tertile_country")
df = add_country_tertiles(df, "age", "age_tertile_country")
df = add_country_tertiles(df, "Hhsize", "hhsize_tertile_country")
df = add_country_tertiles(df, "gender", "gender_tertile_country")

attr_map = {
    "ses": "ses_tertile_country",
    "rural": "rural_tertile_country",
    "education": "education_tertile_country",
    "age": "age_tertile_country",
    "hhsize": "hhsize_tertile_country",
    "gender": "gender_tertile_country"
}

region_order = [
    "Sub-Saharan Africa",
    "Northern Africa and Western Asia",
    "South Asia",
    "Eastern and South-eastern Asia",
    "Europe",
    "Latin America and the Caribbean",
    "Northern America and Oceania"
]
attr_order = list(attr_map.keys())
group_order = ["low", "middle", "high"]

# =========================
# 5. region-level decomposition
# =========================
region_rows = []
region_group_rows = []

for region in region_order:
    sub_region = df[df["analysis_region"] == region].copy()
    if len(sub_region) == 0:
        continue

    for attr, group_col in attr_map.items():
        result, group_df = theil_decompose_by_group(
            sub=sub_region,
            outcome_col="EC2015_pc",
            weight_col="population",
            group_col=group_col
        )

        if result is None:
            region_rows.append({
                "analysis_region": region,
                "attribute": attr,
                "sample_units": 0,
                "population_sum": np.nan,
                "overall": np.nan,
                "between": np.nan,
                "within": np.nan,
                "between_share": np.nan,
                "within_share": np.nan
            })
            continue

        region_rows.append({
            "analysis_region": region,
            "attribute": attr,
            **result
        })

        group_df["analysis_region"] = region
        group_df["attribute"] = attr
        region_group_rows.append(group_df)

region_results = pd.DataFrame(region_rows)
region_group_summary = pd.concat(region_group_rows, ignore_index=True)

region_results["analysis_region"] = pd.Categorical(region_results["analysis_region"], categories=region_order, ordered=True)
region_results["attribute"] = pd.Categorical(region_results["attribute"], categories=attr_order, ordered=True)
region_results = region_results.sort_values(["analysis_region", "attribute"]).reset_index(drop=True)

region_group_summary["analysis_region"] = pd.Categorical(region_group_summary["analysis_region"], categories=region_order, ordered=True)
region_group_summary["attribute"] = pd.Categorical(region_group_summary["attribute"], categories=attr_order, ordered=True)
region_group_summary["group"] = pd.Categorical(region_group_summary["group"], categories=group_order, ordered=True)
region_group_summary = region_group_summary.sort_values(["analysis_region", "attribute", "group"]).reset_index(drop=True)

region_results_csv = os.path.join(region_out_dir, "region_attribute_theil_decomposition.csv")
region_groups_csv = os.path.join(region_out_dir, "region_attribute_group_summary.csv")
region_results.to_csv(region_results_csv, index=False)
region_group_summary.to_csv(region_groups_csv, index=False)

region_between = region_results.pivot(index="analysis_region", columns="attribute", values="between_share")
region_between = region_between.loc[region_order, attr_order]

print("Region between-share table:")
display(region_between.round(3))

# Jupyter display: grouped bar only
fig, ax = plt.subplots(figsize=(12, 5.5))
x = np.arange(len(region_order))
bar_w = 0.13

for k, attr in enumerate(attr_order):
    vals = region_between[attr].values
    ax.bar(x + (k - (len(attr_order)-1)/2) * bar_w, vals, width=bar_w, label=attr)

ax.set_xticks(x)
ax.set_xticklabels(region_order, rotation=45, ha="right")
ax.set_ylabel("between_share")
ax.set_title("Between-group share by region across attributes")
ax.legend(ncol=3, frameon=False)
plt.tight_layout()

region_bar = os.path.join(region_out_dir, "region_between_share_groupedbar.png")
plt.savefig(region_bar, dpi=300, bbox_inches="tight")
plt.show()

for attr in attr_order:
    fig, ax = plt.subplots(figsize=(8, 4.5))
    sub = region_results[region_results["attribute"] == attr].copy()
    sub = sub.set_index("analysis_region").reindex(region_order).reset_index()
    ax.bar(sub["analysis_region"], sub["between_share"])
    ax.set_title(f"Between-group share by region: {attr}")
    ax.set_ylabel("between_share")
    ax.tick_params(axis="x", rotation=45)
    plt.tight_layout()
    fpath = os.path.join(region_out_dir, f"region_between_share_{attr}.png")
    plt.savefig(fpath, dpi=300, bbox_inches="tight")
    plt.close(fig)

# =========================
# 6. country-level decomposition
# representative countries
# =========================
focus_countries = [
    "Nigeria",
    "Egypt",
    "India",
    "China",
    "Indonesia",
    "Germany",
    "France",
    "Brazil",
    "United States",
    "Australia"
]

df_country = df[df["country"].isin(focus_countries)].copy()

country_rows = []
country_group_rows = []

for country in focus_countries:
    sub_country = df_country[df_country["country"] == country].copy()
    if len(sub_country) == 0:
        continue

    for attr, group_col in attr_map.items():
        result, group_df = theil_decompose_by_group(
            sub=sub_country,
            outcome_col="EC2015_pc",
            weight_col="population",
            group_col=group_col
        )

        if result is None:
            country_rows.append({
                "analysis_region": sub_country["analysis_region"].iloc[0],
                "country": country,
                "attribute": attr,
                "sample_units": 0,
                "population_sum": np.nan,
                "overall": np.nan,
                "between": np.nan,
                "within": np.nan,
                "between_share": np.nan,
                "within_share": np.nan
            })
            continue

        country_rows.append({
            "analysis_region": sub_country["analysis_region"].iloc[0],
            "country": country,
            "attribute": attr,
            **result
        })

        group_df["analysis_region"] = sub_country["analysis_region"].iloc[0]
        group_df["country"] = country
        group_df["attribute"] = attr
        country_group_rows.append(group_df)

country_results = pd.DataFrame(country_rows)
country_group_summary = pd.concat(country_group_rows, ignore_index=True)

country_results["country"] = pd.Categorical(country_results["country"], categories=focus_countries, ordered=True)
country_results["attribute"] = pd.Categorical(country_results["attribute"], categories=attr_order, ordered=True)
country_results = country_results.sort_values(["country", "attribute"]).reset_index(drop=True)

country_group_summary["country"] = pd.Categorical(country_group_summary["country"], categories=focus_countries, ordered=True)
country_group_summary["attribute"] = pd.Categorical(country_group_summary["attribute"], categories=attr_order, ordered=True)
country_group_summary["group"] = pd.Categorical(country_group_summary["group"], categories=group_order, ordered=True)
country_group_summary = country_group_summary.sort_values(["country", "attribute", "group"]).reset_index(drop=True)

country_results_csv = os.path.join(country_out_dir, "country_attribute_theil_decomposition.csv")
country_groups_csv = os.path.join(country_out_dir, "country_attribute_group_summary.csv")
country_results.to_csv(country_results_csv, index=False)
country_group_summary.to_csv(country_groups_csv, index=False)

country_between = country_results.pivot(index="country", columns="attribute", values="between_share")
country_between = country_between.loc[focus_countries, attr_order]

print("\nCountry between-share table:")
display(country_between.round(3))

# Jupyter display: grouped bar only
fig, ax = plt.subplots(figsize=(12, 5.5))
x = np.arange(len(focus_countries))
bar_w = 0.13

for k, attr in enumerate(attr_order):
    vals = country_between[attr].values
    ax.bar(x + (k - (len(attr_order)-1)/2) * bar_w, vals, width=bar_w, label=attr)

ax.set_xticks(x)
ax.set_xticklabels(focus_countries, rotation=45, ha="right")
ax.set_ylabel("between_share")
ax.set_title("Between-group share by country across attributes")
ax.legend(ncol=3, frameon=False)
plt.tight_layout()

country_bar = os.path.join(country_out_dir, "country_between_share_groupedbar.png")
plt.savefig(country_bar, dpi=300, bbox_inches="tight")
plt.show()

for attr in attr_order:
    fig, ax = plt.subplots(figsize=(8, 4.5))
    sub = country_results[country_results["attribute"] == attr].copy()
    sub = sub.set_index("country").reindex(focus_countries).reset_index()
    ax.bar(sub["country"], sub["between_share"])
    ax.set_title(f"Between-group share by country: {attr}")
    ax.set_ylabel("between_share")
    ax.tick_params(axis="x", rotation=45)
    plt.tight_layout()
    fpath = os.path.join(country_out_dir, f"country_between_share_{attr}.png")
    plt.savefig(fpath, dpi=300, bbox_inches="tight")
    plt.close(fig)

print("\nSaved region files:")
print(region_results_csv)
print(region_groups_csv)
print(region_bar)

print("\nSaved country files:")
print(country_results_csv)
print(country_groups_csv)
print(country_bar)

print(f"\nAll outputs saved under: {base_out_dir}")